In [1]:
# =====================================================================
# PART 1 OF 2 : ECDF AND BOX-PLOT FEATURE DISTRIBUTIONS
#
# Input:
#     /content/drive/MyDrive/Objective2/
#
# Output:
#     /content/drive/MyDrive/Objective2/ECDF_Boxplots/
#
# This notebook performs data loading, preprocessing, and generates
# ECDF and box plots for every retained feature.
#
# It also saves the cleaned dataset so the companion notebook
# (classification.ipynb) can train the models without repeating
# the preprocessing.
#
# Figures:
#     - ECDF plots: full empirical cumulative distributions
#     - Box plots: median, IQR and whiskers; extreme fliers hidden visually
#     - Benign, DNSCat2, DNSTT and Iodine use fixed class colours
#     - No titles
#     - PDF at 600 dpi
# =====================================================================


In [2]:
# =====================================================================
# 1. MOUNT GOOGLE DRIVE
# =====================================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# =====================================================================
# 2. IMPORT LIBRARIES
# =====================================================================

from pathlib import Path
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from IPython.display import display

warnings.filterwarnings("ignore")


In [4]:
# =====================================================================
# 3. GLOBAL SETTINGS
# =====================================================================

RANDOM_STATE = 42

# ECDF uses all finite observations (no sampling and no smoothing).
# Box plots use all finite observations as well.
#
# Extreme box-plot fliers are hidden only in the figure so that a few
# unusually large/small values do not obscure the central distribution.
# No observations are deleted from the dataset.
SHOW_BOX_FLIERS = False

# Remove an existing output folder before creating new results.
RESET_OUTPUT_FOLDER = True


In [5]:
# =====================================================================
# 4. FONT AND FIGURE SETTINGS
# =====================================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": [
        "DejaVu Serif",
        "Computer Modern",
        "Times New Roman"
    ],
    "font.size": 20,
    "axes.titlesize": 22,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 14,
    "figure.titlesize": 22
})


# Fixed colours for the four traffic classes.
# DNSTT and Iodine are deliberately chosen to be visually distinct.
CLASS_COLOURS = {
    "Benign": "#2F6690",    # muted blue
    "DNSCat2": "#8B5E3C",   # coffee brown
    "DNSTT": "#D95F02",     # strong orange
    "Iodine": "#6A3D9A"     # purple
}

LABEL_ORDER = [
    "Benign",
    "DNSCat2",
    "DNSTT",
    "Iodine"
]


# Colours used for ROC curves
ROC_COLOURS = {
    "Logistic Regression": "#2F6690",
    "Decision Tree": "#8B5E3C",
    "Random Forest": "#9B1B30",
    "SVM": "#D36C9D"
}


In [6]:
# =====================================================================
# 5. INPUT AND OUTPUT DIRECTORIES
# =====================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/Objective2"
)

OUTPUT_DIR = BASE_DIR / "ECDF_Boxplots"

ECDF_DIR = OUTPUT_DIR / "ECDF"
BOXPLOT_DIR = OUTPUT_DIR / "Boxplots"
CLEANED_DATA_DIR = OUTPUT_DIR / "cleaned_data"
MODEL_DIR = OUTPUT_DIR / "saved_models"


if RESET_OUTPUT_FOLDER and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)


for directory in [
    OUTPUT_DIR,
    ECDF_DIR,
    BOXPLOT_DIR,
    CLEANED_DATA_DIR,
    MODEL_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("Input directory :", BASE_DIR)
print("Output directory:", OUTPUT_DIR)


Input directory : /content/drive/MyDrive/Objective2
Output directory: /content/drive/MyDrive/Objective2/ECDF_Boxplots


In [7]:
# =====================================================================
# 6. LOCATE THE FOUR CSV FILES
# =====================================================================

csv_files = sorted(
    BASE_DIR.glob("*.csv")
)


if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files were found inside:\n{BASE_DIR}"
    )


print("\nCSV files found:")

for file_path in csv_files:
    print(" -", file_path.name)


def clean_filename(filename):
    """
    Convert a filename to lowercase alphanumeric text.
    """

    return re.sub(
        r"[^a-z0-9]",
        "",
        filename.lower()
    )


def find_csv(keyword):
    """
    Find a CSV file whose name contains the keyword.
    """

    cleaned_keyword = clean_filename(
        keyword
    )

    matches = []

    for file_path in csv_files:

        cleaned_file_name = clean_filename(
            file_path.stem
        )

        if cleaned_keyword in cleaned_file_name:
            matches.append(file_path)


    if len(matches) == 0:

        raise FileNotFoundError(
            f"No CSV file matching '{keyword}' was found "
            f"inside {BASE_DIR}"
        )


    if len(matches) > 1:

        print(
            f"Warning: multiple files matched '{keyword}'. "
            f"Using: {matches[0].name}"
        )


    return matches[0]


FILES = {
    "Benign": find_csv("benign"),
    "DNSCat2": find_csv("dnscat"),
    "DNSTT": find_csv("dnstt"),
    "Iodine": find_csv("iodine")
}


print("\nSelected files:")

for dataset_name, file_path in FILES.items():

    print(
        f"{dataset_name:8s}: {file_path.name}"
    )


CSV files found:
 - Benign.csv
 - Dnscat2.csv
 - Dnstt.csv
 - Iodine.csv

Selected files:
Benign  : Benign.csv
DNSCat2 : Dnscat2.csv
DNSTT   : Dnstt.csv
Iodine  : Iodine.csv


In [8]:
# =====================================================================
# 7. HELPER FUNCTIONS FOR COLUMN IDENTIFICATION
# =====================================================================

def normalise_column_name(column_name):
    """
    Remove spaces and punctuation from a column name.
    """

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(column_name).lower()
    )


def find_column_case_insensitive(
    dataframe,
    possible_names
):
    """
    Find a dataframe column using several possible names.
    """

    available_columns = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }


    for possible_name in possible_names:

        normalised_name = normalise_column_name(
            possible_name
        )

        if normalised_name in available_columns:

            return available_columns[
                normalised_name
            ]


    return None

In [9]:
# =====================================================================
# 8. READ ALL FOUR CSV FILES
# =====================================================================

frames = []
raw_file_summary = []


for dataset_name, file_path in FILES.items():

    print(
        f"\nReading {dataset_name}: "
        f"{file_path.name}"
    )

    dataframe = pd.read_csv(
        file_path,
        low_memory=False
    )


    # Strip spaces from column names.
    dataframe.columns = [
        str(column).strip()
        for column in dataframe.columns
    ]


    # Find and standardise the Class column.
    class_column = find_column_case_insensitive(
        dataframe,
        ["Class"]
    )


    if class_column is None:

        raise ValueError(
            f"Class column was not found in "
            f"{file_path.name}"
        )


    if class_column != "Class":

        dataframe = dataframe.rename(
            columns={
                class_column: "Class"
            }
        )


    # Find and standardise the Label column.
    label_column = find_column_case_insensitive(
        dataframe,
        ["Label"]
    )


    if label_column is None:

        dataframe["Label"] = "Benign"

    elif label_column != "Label":

        dataframe = dataframe.rename(
            columns={
                label_column: "Label"
            }
        )


    dataframe["DatasetSource"] = dataset_name


    numeric_dataframe = dataframe.select_dtypes(
        include=[np.number]
    )


    if numeric_dataframe.shape[1] > 0:

        infinite_values = int(
            np.isinf(
                numeric_dataframe.to_numpy(
                    dtype=float
                )
            ).sum()
        )

    else:

        infinite_values = 0


    raw_file_summary.append({
        "Dataset": dataset_name,
        "Filename": file_path.name,
        "Rows": len(dataframe),
        "Columns": dataframe.shape[1],
        "MissingValues":
            int(dataframe.isna().sum().sum()),
        "InfiniteValues":
            infinite_values,
        "ExactFullRowDuplicates":
            int(dataframe.duplicated().sum())
    })


    frames.append(dataframe)


raw_summary_df = pd.DataFrame(
    raw_file_summary
)


raw_summary_df.to_csv(
    OUTPUT_DIR / "raw_file_summary.csv",
    index=False
)


print("\nRaw-file summary:")

display(
    raw_summary_df
)


raw_data = pd.concat(
    frames,
    ignore_index=True,
    sort=False
)


print(
    "\nCombined raw-data shape:",
    raw_data.shape
)


Reading Benign: Benign.csv

Reading DNSCat2: Dnscat2.csv

Reading DNSTT: Dnstt.csv

Reading Iodine: Iodine.csv

Raw-file summary:


,Dataset,Filename,Rows,Columns,MissingValues,InfiniteValues,ExactFullRowDuplicates
0,Benign,Benign.csv,95234,37,0,0,0
1,DNSCat2,Dnscat2.csv,105009,37,0,0,0
2,DNSTT,Dnstt.csv,90334,37,0,0,0
3,Iodine,Iodine.csv,140000,37,0,0,0



Combined raw-data shape: (430577, 37)


In [10]:
# =====================================================================
# 9. NORMALISE CLASS AND TOOL LABELS
# =====================================================================

raw_data["Class"] = (
    raw_data["Class"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "normal": "benign",
        "attack": "malicious"
    })
    .str.title()
)


def standardise_tool_label(value):
    """
    Standardise tool-name spellings.
    """

    cleaned_value = str(value).strip().lower()


    if "benign" in cleaned_value:
        return "Benign"


    if "dnscat" in cleaned_value:
        return "DNSCat2"


    if "dnstt" in cleaned_value:
        return "DNSTT"


    if "iodine" in cleaned_value:
        return "Iodine"


    return str(value).strip()


raw_data["Label"] = (
    raw_data["Label"]
    .apply(standardise_tool_label)
)


# Force every benign row to use the Benign tool label.
raw_data.loc[
    raw_data["Class"]
    .str.lower()
    .eq("benign"),
    "Label"
] = "Benign"


print("\nClass values:")

print(
    raw_data["Class"]
    .value_counts(dropna=False)
)


print("\nTool-label values:")

print(
    raw_data["Label"]
    .value_counts(dropna=False)
)


Class values:
Class
Malicious    335343
Benign        95234
Name: count, dtype: int64

Tool-label values:
Label
Iodine     140000
DNSCat2    105009
Benign      95234
DNSTT       90334
Name: count, dtype: int64


In [11]:
# =====================================================================
# 10. REMOVE SOURCE IP, DESTINATION IP, PORTS AND TIMESTAMP
# =====================================================================

columns_to_exclude = [
    "SourceIP",
    "Source IP",
    "SrcIP",
    "Src IP",

    "DestinationIP",
    "Destination IP",
    "DestIP",
    "DstIP",
    "Dst IP",

    "SourcePort",
    "Source Port",
    "SrcPort",
    "Src Port",

    "DestinationPort",
    "Destination Port",
    "DestPort",
    "DstPort",
    "Dst Port",

    "TimeStamp",
    "Timestamp",
    "Time Stamp",
    "DateTime",
    "Date Time",
    "CaptureTime",
    "Capture Time",

    "DatasetSource"
]


normalised_excluded_names = {
    normalise_column_name(column)
    for column in columns_to_exclude
}


actual_columns_to_remove = [
    column
    for column in raw_data.columns
    if normalise_column_name(column)
    in normalised_excluded_names
]


print("\nColumns removed:")

for column in actual_columns_to_remove:
    print(" -", column)


data = raw_data.drop(
    columns=actual_columns_to_remove,
    errors="ignore"
).copy()


Columns removed:
 - SourceIP
 - DestinationIP
 - SourcePort
 - DestinationPort
 - TimeStamp
 - DatasetSource


In [12]:
# =====================================================================
# 11. IDENTIFY FEATURE COLUMNS
# =====================================================================

TARGET_COLUMNS = [
    "Class",
    "Label"
]


FEATURE_COLUMNS = [
    column
    for column in data.columns
    if column not in TARGET_COLUMNS
]


print(
    "\nNumber of retained model features:",
    len(FEATURE_COLUMNS)
)


print("\nFeature columns:")

for feature_number, feature in enumerate(
    FEATURE_COLUMNS,
    start=1
):

    print(
        f"{feature_number:02d}. {feature}"
    )


Number of retained model features: 29

Feature columns:
01. Duration
02. FlowBytesSent
03. FlowSentRate
04. FlowBytesReceived
05. FlowReceivedRate
06. PacketLengthVariance
07. PacketLengthStandardDeviation
08. PacketLengthMean
09. PacketLengthMedian
10. PacketLengthMode
11. PacketLengthSkewFromMedian
12. PacketLengthSkewFromMode
13. PacketLengthCoefficientofVariation
14. PacketTimeVariance
15. PacketTimeStandardDeviation
16. PacketTimeMean
17. PacketTimeMedian
18. PacketTimeMode
19. PacketTimeSkewFromMedian
20. PacketTimeSkewFromMode
21. PacketTimeCoefficientofVariation
22. ResponseTimeTimeVariance
23. ResponseTimeTimeStandardDeviation
24. ResponseTimeTimeMean
25. ResponseTimeTimeMedian
26. ResponseTimeTimeMode
27. ResponseTimeTimeSkewFromMedian
28. ResponseTimeTimeSkewFromMode
29. ResponseTimeTimeCoefficientofVariation


In [13]:
# =====================================================================
# 12. CONVERT FEATURES TO NUMERIC
# =====================================================================

for feature in FEATURE_COLUMNS:

    data[feature] = pd.to_numeric(
        data[feature],
        errors="coerce"
    )


# Convert infinity to missing values.
data[FEATURE_COLUMNS] = (
    data[FEATURE_COLUMNS]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

In [14]:
# =====================================================================
# 13. HANDLE MISSING VALUES
# =====================================================================

missing_values_before = int(
    data[FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)


print(
    "\nMissing values before imputation:",
    missing_values_before
)


for feature in FEATURE_COLUMNS:

    if data[feature].isna().any():

        feature_mean = data[feature].mean()


        if pd.isna(feature_mean):
            feature_mean = 0.0


        data[feature] = (
            data[feature]
            .fillna(feature_mean)
        )


missing_values_after = int(
    data[FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)


print(
    "Missing values after imputation :",
    missing_values_after
)


Missing values before imputation: 0
Missing values after imputation : 0


In [15]:
# =====================================================================
# 14. REMOVE DUPLICATES AFTER IDENTIFIERS ARE REMOVED
# =====================================================================

rows_before_duplicate_removal = len(
    data
)


data = (
    data
    .drop_duplicates()
    .reset_index(drop=True)
)


rows_after_duplicate_removal = len(
    data
)


duplicates_removed = (
    rows_before_duplicate_removal
    - rows_after_duplicate_removal
)


print(
    "\nRows before duplicate removal:",
    rows_before_duplicate_removal
)

print(
    "Rows after duplicate removal :",
    rows_after_duplicate_removal
)

print(
    "Duplicate rows removed       :",
    duplicates_removed
)


Rows before duplicate removal: 430577
Rows after duplicate removal : 407623
Duplicate rows removed       : 22954


In [16]:
# =====================================================================
# 15. REMOVE CONTRADICTORY FEATURE VECTORS
# =====================================================================
#
# A contradictory feature vector has exactly the same model features
# but different class or tool labels.
# Such rows can create unavoidable label conflict.
# =====================================================================

data["FeatureVectorHash"] = (
    pd.util.hash_pandas_object(
        data[FEATURE_COLUMNS],
        index=False
    )
    .astype(str)
)


data["CombinedTarget"] = (
    data["Class"].astype(str)
    + "_"
    + data["Label"].astype(str)
)


target_count_per_hash = (
    data
    .groupby("FeatureVectorHash")
    ["CombinedTarget"]
    .nunique()
)


contradictory_hashes = (
    target_count_per_hash[
        target_count_per_hash > 1
    ]
    .index
)


contradictory_rows = int(
    data["FeatureVectorHash"]
    .isin(contradictory_hashes)
    .sum()
)


print(
    "\nContradictory feature groups:",
    len(contradictory_hashes)
)

print(
    "Rows in contradictory groups:",
    contradictory_rows
)


if contradictory_rows > 0:

    data = data.loc[
        ~data["FeatureVectorHash"]
        .isin(contradictory_hashes)
    ].copy()


data = (
    data
    .drop(
        columns=["CombinedTarget"]
    )
    .reset_index(drop=True)
)


print(
    "Rows retained after contradiction removal:",
    len(data)
)


Contradictory feature groups: 1
Rows in contradictory groups: 2
Rows retained after contradiction removal: 407621


In [17]:
# =====================================================================
# 16. POST-PREPROCESSING SUMMARY
# =====================================================================

post_preprocessing_summary = pd.DataFrame([{
    "RawRows":
        len(raw_data),

    "RowsBeforeDuplicateRemoval":
        rows_before_duplicate_removal,

    "RowsAfterDuplicateRemoval":
        rows_after_duplicate_removal,

    "DuplicatesRemoved":
        duplicates_removed,

    "ContradictoryRowsRemoved":
        contradictory_rows,

    "FinalRows":
        len(data),

    "RetainedFeatures":
        len(FEATURE_COLUMNS),

    "MissingValues":
        int(
            data[FEATURE_COLUMNS]
            .isna()
            .sum()
            .sum()
        ),

    "InfiniteValues":
        int(
            np.isinf(
                data[FEATURE_COLUMNS]
                .to_numpy(dtype=float)
            ).sum()
        )
}])


post_preprocessing_summary.to_csv(
    OUTPUT_DIR /
    "post_preprocessing_summary.csv",
    index=False
)


print("\nPost-preprocessing summary:")

display(
    post_preprocessing_summary
)


Post-preprocessing summary:


,RawRows,RowsBeforeDuplicateRemoval,RowsAfterDuplicateRemoval,DuplicatesRemoved,ContradictoryRowsRemoved,FinalRows,RetainedFeatures,MissingValues,InfiniteValues
0,430577,430577,407623,22954,2,407621,29,0,0


In [18]:
# =====================================================================
# 17. SAVE FEATURE STATISTICS
# =====================================================================

feature_summary = pd.DataFrame(
    index=FEATURE_COLUMNS
)


feature_summary["Minimum"] = (
    data[FEATURE_COLUMNS]
    .min()
)


feature_summary["Maximum"] = (
    data[FEATURE_COLUMNS]
    .max()
)


feature_summary["Mean"] = (
    data[FEATURE_COLUMNS]
    .mean()
)


feature_summary["Median"] = (
    data[FEATURE_COLUMNS]
    .median()
)


feature_summary["StandardDeviation"] = (
    data[FEATURE_COLUMNS]
    .std()
)


feature_summary["Skewness"] = (
    data[FEATURE_COLUMNS]
    .skew()
)


feature_summary["UniqueValues"] = (
    data[FEATURE_COLUMNS]
    .nunique()
)


feature_summary["Minus1Count"] = (
    data[FEATURE_COLUMNS] == -1
).sum()


feature_summary["Minus10Count"] = (
    data[FEATURE_COLUMNS] == -10
).sum()


feature_summary.index.name = "Feature"


feature_summary.reset_index().to_csv(
    OUTPUT_DIR /
    "post_preprocessing_feature_summary.csv",
    index=False
)

In [19]:
# =====================================================================
# 18. GENERATE ECDF AND BOX PLOTS FOR EVERY FEATURE
# =====================================================================
#
# ECDF:
#   - Uses every finite observation.
#   - No KDE bandwidth and no smoothing.
#   - The x-axis is limited using per-class 0.5th and 99.5th percentiles
#     only for readability.
#   - The ECDF itself is computed from the FULL class distribution, so
#     cumulative probabilities are not renormalised after visual clipping.
#
# Box plot:
#   - Uses every finite observation.
#   - Shows median, Q1, Q3 and standard 1.5*IQR whiskers.
#   - Extreme fliers are hidden visually when SHOW_BOX_FLIERS=False.
#   - No rows are removed from the dataset.
# =====================================================================


def empirical_cdf(values):
    """
    Return sorted finite values and their empirical cumulative
    probabilities.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    values = np.sort(
        values
    )

    if len(values) == 0:
        return values, np.array([])

    cumulative_probability = (
        np.arange(
            1,
            len(values) + 1
        )
        /
        len(values)
    )

    return (
        values,
        cumulative_probability
    )


print("\nGenerating ECDF and box plots...")


for feature_number, feature in enumerate(
    FEATURE_COLUMNS,
    start=1
):

    all_values = (
        data[feature]
        .to_numpy(dtype=float)
    )

    all_values = all_values[
        np.isfinite(all_values)
    ]


    if len(all_values) < 2:

        print(
            f"Skipping {feature}: "
            "insufficient values"
        )

        continue


    # ---------------------------------------------------------------
    # Compute robust ECDF display limits PER CLASS, then combine them.
    # This prevents one extreme outlier from stretching the x-axis
    # while preserving the ECDF probabilities from the complete data.
    # ---------------------------------------------------------------

    per_class_lower_limits = []
    per_class_upper_limits = []


    for label in LABEL_ORDER:

        class_values = (
            data.loc[
                data["Label"].eq(label),
                feature
            ]
            .to_numpy(dtype=float)
        )

        class_values = class_values[
            np.isfinite(class_values)
        ]


        if len(class_values) < 2:
            continue


        per_class_lower_limits.append(
            np.quantile(
                class_values,
                0.005
            )
        )

        per_class_upper_limits.append(
            np.quantile(
                class_values,
                0.995
            )
        )


    if len(per_class_lower_limits) > 0:

        lower_limit = min(
            per_class_lower_limits
        )

        upper_limit = max(
            per_class_upper_limits
        )

    else:

        lower_limit = np.quantile(
            all_values,
            0.005
        )

        upper_limit = np.quantile(
            all_values,
            0.995
        )


    if (
        not np.isfinite(lower_limit)
        or not np.isfinite(upper_limit)
    ):

        lower_limit = np.min(
            all_values
        )

        upper_limit = np.max(
            all_values
        )


    if lower_limit == upper_limit:

        lower_limit -= 0.5
        upper_limit += 0.5


    safe_feature_name = re.sub(
        r"[^A-Za-z0-9_-]",
        "_",
        feature
    )


    # ===============================================================
    # A. ECDF PLOT
    # ===============================================================

    ecdf_figure, ecdf_axis = plt.subplots(
        figsize=(10.0, 7.0)
    )

    plotted_curve_count = 0


    for label in LABEL_ORDER:

        label_values = (
            data.loc[
                data["Label"].eq(label),
                feature
            ]
            .to_numpy(dtype=float)
        )


        x_values, y_values = empirical_cdf(
            label_values
        )


        if len(x_values) == 0:
            continue


        ecdf_axis.step(
            x_values,
            y_values,
            where="post",
            color=CLASS_COLOURS[label],
            linewidth=2.5,
            label=label
        )

        plotted_curve_count += 1


    ecdf_axis.set_xlabel(
        feature
    )

    ecdf_axis.set_ylabel(
        "Cumulative Probability"
    )

    ecdf_axis.set_xlim(
        lower_limit,
        upper_limit
    )

    ecdf_axis.set_ylim(
        0.0,
        1.0
    )

    ecdf_axis.grid(
        True,
        axis="both",
        linestyle="--",
        linewidth=0.7,
        alpha=0.30
    )

    ecdf_axis.spines["top"].set_visible(
        False
    )

    ecdf_axis.spines["right"].set_visible(
        False
    )


    if plotted_curve_count > 0:

        ecdf_axis.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.20),
            ncol=4,
            frameon=False,
            columnspacing=1.8,
            handlelength=2.5
        )


    ecdf_figure.subplots_adjust(
        bottom=0.27
    )


    ecdf_output_path = (
        ECDF_DIR
        /
        f"{feature_number:02d}_"
        f"{safe_feature_name}_ecdf.pdf"
    )


    ecdf_figure.savefig(
        ecdf_output_path,
        format="pdf",
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(
        ecdf_figure
    )


    # ===============================================================
    # B. BOX PLOT
    # ===============================================================

    box_values = []
    box_labels = []
    box_colours = []


    for label in LABEL_ORDER:

        label_values = (
            data.loc[
                data["Label"].eq(label),
                feature
            ]
            .to_numpy(dtype=float)
        )

        label_values = label_values[
            np.isfinite(label_values)
        ]


        if len(label_values) == 0:
            continue


        box_values.append(
            label_values
        )

        box_labels.append(
            label
        )

        box_colours.append(
            CLASS_COLOURS[label]
        )


    if len(box_values) > 0:

        box_figure, box_axis = plt.subplots(
            figsize=(10.0, 7.0)
        )


        boxplot = box_axis.boxplot(
            box_values,
            labels=box_labels,
            patch_artist=True,
            showfliers=SHOW_BOX_FLIERS,
            widths=0.60,
            whis=1.5,
            medianprops={
                "color": "black",
                "linewidth": 2.2
            },
            whiskerprops={
                "linewidth": 1.6
            },
            capprops={
                "linewidth": 1.6
            },
            boxprops={
                "linewidth": 1.6
            }
        )


        for patch, colour in zip(
            boxplot["boxes"],
            box_colours
        ):

            patch.set_facecolor(
                colour
            )

            patch.set_alpha(
                0.80
            )


        box_axis.set_xlabel(
            "Traffic Class"
        )

        box_axis.set_ylabel(
            feature
        )

        box_axis.grid(
            True,
            axis="y",
            linestyle="--",
            linewidth=0.7,
            alpha=0.30
        )

        box_axis.spines["top"].set_visible(
            False
        )

        box_axis.spines["right"].set_visible(
            False
        )


        # A colour legend is included below the graph so the box-plot
        # presentation is consistent with the ECDF figures.
        legend_handles = [
            Patch(
                facecolor=CLASS_COLOURS[label],
                edgecolor="black",
                label=label,
                alpha=0.80
            )
            for label in box_labels
        ]


        box_axis.legend(
            handles=legend_handles,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.20),
            ncol=len(box_labels),
            frameon=False,
            columnspacing=1.8,
            handlelength=1.8
        )


        box_figure.subplots_adjust(
            bottom=0.27
        )


        box_output_path = (
            BOXPLOT_DIR
            /
            f"{feature_number:02d}_"
            f"{safe_feature_name}_boxplot.pdf"
        )


        box_figure.savefig(
            box_output_path,
            format="pdf",
            dpi=600,
            bbox_inches="tight"
        )

        plt.close(
            box_figure
        )


    print(
        f"{feature_number:02d}/"
        f"{len(FEATURE_COLUMNS)} "
        f"saved: "
        f"{safe_feature_name}_ecdf.pdf + "
        f"{safe_feature_name}_boxplot.pdf"
    )


print(
    "\nNumber of ECDF PDFs:",
    len(list(ECDF_DIR.glob("*.pdf")))
)

print(
    "Number of box-plot PDFs:",
    len(list(BOXPLOT_DIR.glob("*.pdf")))
)



Generating ECDF and box plots...
01/29 saved: Duration_ecdf.pdf + Duration_boxplot.pdf
02/29 saved: FlowBytesSent_ecdf.pdf + FlowBytesSent_boxplot.pdf
03/29 saved: FlowSentRate_ecdf.pdf + FlowSentRate_boxplot.pdf
04/29 saved: FlowBytesReceived_ecdf.pdf + FlowBytesReceived_boxplot.pdf
05/29 saved: FlowReceivedRate_ecdf.pdf + FlowReceivedRate_boxplot.pdf
06/29 saved: PacketLengthVariance_ecdf.pdf + PacketLengthVariance_boxplot.pdf
07/29 saved: PacketLengthStandardDeviation_ecdf.pdf + PacketLengthStandardDeviation_boxplot.pdf
08/29 saved: PacketLengthMean_ecdf.pdf + PacketLengthMean_boxplot.pdf
09/29 saved: PacketLengthMedian_ecdf.pdf + PacketLengthMedian_boxplot.pdf
10/29 saved: PacketLengthMode_ecdf.pdf + PacketLengthMode_boxplot.pdf
11/29 saved: PacketLengthSkewFromMedian_ecdf.pdf + PacketLengthSkewFromMedian_boxplot.pdf
12/29 saved: PacketLengthSkewFromMode_ecdf.pdf + PacketLengthSkewFromMode_boxplot.pdf
13/29 saved: PacketLengthCoefficientofVariation_ecdf.pdf + PacketLengthCoefficie

In [20]:
# =====================================================================
# 19. SAVE THE CLEANED COMBINED DATASET (HAND-OFF TO classification.ipynb)
# =====================================================================
#
# The cleaned, deduplicated, contradiction-free dataset is saved once
# here so classification.ipynb can load it directly and both notebooks
# are guaranteed to operate on identical rows.
# =====================================================================

data.to_csv(
    CLEANED_DATA_DIR /
    "combined_cleaned_data.csv.gz",
    index=False,
    compression="gzip"
)

print(
    "Saved cleaned dataset:",
    CLEANED_DATA_DIR / "combined_cleaned_data.csv.gz"
)

print(
    "Rows:",
    len(data),
    "| Features:",
    len(FEATURE_COLUMNS)
)


Saved cleaned dataset: /content/drive/MyDrive/Objective2/ECDF_Boxplots/cleaned_data/combined_cleaned_data.csv.gz
Rows: 407621 | Features: 29


In [21]:
# =====================================================================
# 20. CREATE ZIP FILE CONTAINING ECDF AND BOX-PLOT FIGURES
# =====================================================================

plots_zip_path = shutil.make_archive(
    base_name=str(
        BASE_DIR /
        "ecdf_boxplots_600dpi"
    ),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

print("\nECDF + box-plot ZIP file:")
print(plots_zip_path)



ECDF + box-plot ZIP file:
/content/drive/MyDrive/Objective2/ecdf_boxplots_600dpi.zip
